# Rung H4 without the channel of the next packet

A network server sets the data rate and transmit power of a link before its next uplink, and the end device then picks the channel of that uplink at random among the enabled ones. Rung H4 uses that channel in four inputs, the ones rung H2 adds: the channel frequency and the same-channel history (the last path loss, the mean of the last 5 packets and the time since the last packet on that channel). Without them, rung H4 holds what a server knows when it decides. This notebook repeats on the remaining 21 inputs, at the 99 % target and with the hyperparameters tuned on the full rung H4 in notebook 13:

* the base predictors of notebook 16, a LightGBM regression and a LightGBM quantile model, with its five calibrations of the shift from the scores of validation folds 0 to 4, scored on the hold-out: coverage overall and in the lowest state, mean margin above the anchor, pinball loss;
* the matched regime of notebook 16: one margin calibrated on the SF7-10 validation packets;
* the comparison of notebook 17 with the estimators of adaptive data rate schemes on the link's earlier packets at the spreading factor of the predicted packet: the four filters and the best of the last 20 packets, with one shift per spreading factor x working hours and with adaptive conformal inference per link, and the differences in pinball loss with 95 % intervals from resampling days.

Writes `CV_Results/server_side_margins.csv`, `server_side_matched_regime.csv` and `server_side_adr_estimators.csv`.

In [1]:
import json
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from history_features import RUNGS
from margin_tools import pinball, shift_pooled, shift_groups, shift_weighted, aci, spread_groups, day_codes, boot_stat

RANDOM_STATE = 42
TAU = 0.99
HALF_LIFE = 30          # days, recency-weighted shift
GAMMA_DIVISOR = 20      # adaptive conformal inference: step size (1 - target) / 20
CHANNEL = ["frequency", "r_c1", "r_mc5", "cgap"]       # channel frequency and same-channel history of the predicted packet

In [2]:
# The rows, folds and inputs of notebooks 13 to 17, rung H4 without the channel inputs
train = pd.read_csv("Data_Files/history_train.csv")
test = pd.read_csv("Data_Files/history_test.csv")
d = pd.concat([train, test], ignore_index=True)
d["t"] = pd.to_datetime(d["t"], utc=True)
d = d.sort_values("t").reset_index(drop=True)
fold = d["fold"].to_numpy()
H = {k.split()[0]: v for k, v in RUNGS.items()}
inputs = [c for c in H["H4"] if c not in CHANNEL]
assert len(inputs) == len(H["H4"]) - len(CHANNEL) and set(CHANNEL) == set(H["H2"]) - set(H["H1"])
ok = (d["y"].notna() & d[H["H7"]].notna().all(axis=1)).to_numpy()          # an anchor and a complete input
PL, a, y = d["PL"].to_numpy(), d["a"].to_numpy(), d["y"].to_numpy()
X_all = d[H["H7"]].copy()
X = X_all.fillna(X_all[ok & (fold < 9)].median())[inputs].to_numpy()
best = json.load(open("CV_Results/history_best_params.json"))["LGBM"]
link, sf, work, s20 = d["device_id"].to_numpy(), d["SF"].to_numpy().astype(int), d["work"].to_numpy().astype(int), d["s20"].to_numpy()
day = day_codes(d["t"])
val, ev = ok & (fold >= 0) & (fold < 9), ok & (fold == 9)
cal, te = np.flatnonzero(val), np.flatnonzero(ev)
print(f"inputs: {len(inputs)} of the {len(H['H4'])} of rung H4; validation folds 0-4: {val.sum():,}; hold-out: {ev.sum():,}")

inputs: 21 of the 25 of rung H4; validation folds 0-4: 1,773,285; hold-out: 531,983


In [3]:
# Base predictors: out-of-fold for validation folds 0-4 (each fitted on the folds before it) and a fit on all training rows for the hold-out
def oof_and_test(make):
    p = np.full(len(d), np.nan)
    for k in range(5):
        tr, va = ok & (fold < k), ok & (fold == k)
        p[va] = make().fit(X[tr], y[tr]).predict(X[va])
    tr = ok & (fold < 9)
    p[ev] = make().fit(X[tr], y[tr]).predict(X[ev])
    return p

rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
params = dict(subsample_freq=1, random_state=RANDOM_STATE, n_jobs=8, verbose=-1, **best)      # 8 threads: LightGBM stalls when oversubscribed
t0 = time.time()
base = {"reg": oof_and_test(lambda: lgb.LGBMRegressor(**params)),
        "q": oof_and_test(lambda: lgb.LGBMRegressor(objective="quantile", alpha=TAU, **params))}
print(f"one regression and one quantile model, each fitted six times: {time.time() - t0:.0f} s")
print(f"regression RMSE: out of fold {rmse((y - base['reg'])[val]):.3f} dB, hold-out {rmse((y - base['reg'])[ev]):.3f} dB")

one regression and one quantile model, each fitted six times: 302 s
regression RMSE: out of fold 2.728 dB, hold-out 1.964 dB


In [4]:
# Margins above the anchor on the hold-out, the shift calibrated five ways on the scores of validation folds 0-4: coverage overall and in the
# lowest state with 1000 packets or more (working hours, recent-spread tercile, spreading factor, link), mean margin and pinball loss
cut = np.quantile(s20[cal], [1 / 3, 2 / 3])

def state_codes(rows):
    g1 = (sf[rows] - 7) * 2 + work[rows]                    # spreading factor x working hours
    return g1, g1 * 3 + spread_groups(s20[rows], cut)       # and the same split by recent-spread tercile

def calibrate(b):
    """Base threshold b plus a shift from the scores of the validation folds, five ways, for the hold-out."""
    score = y[cal] - b[cal]
    g1_cal, g2_cal = state_codes(cal)
    g1_te, g2_te = state_codes(te)
    age = (d["t"].iloc[te].min() - d["t"].iloc[cal]).dt.total_seconds().to_numpy() / 86400
    out = {"pooled": b[te] + shift_pooled(score, TAU),
           "SF x hours": b[te] + shift_groups(score, g1_cal, g1_te, TAU),
           "state table": b[te] + shift_groups(score, g2_cal, g2_te, TAU),
           "recency": b[te] + shift_weighted(score, age, TAU, HALF_LIFE)}
    m = np.empty(len(te))
    for l in np.unique(link[te]):                           # one adaptive sequence per link, in time order
        i = np.flatnonzero(link[te] == l)
        m[i] = aci(b[te][i], y[te][i], score, TAU, (1 - TAU) / GAMMA_DIVISOR)
    out["ACI"] = m
    return out

states = {"hours": work[te], "spread": spread_groups(s20[te], cut), "SF": sf[te], "link": link[te]}
rows = []
for kind in ("reg", "q"):
    for name, m in calibrate(base[kind]).items():
        miss = y[te] > m
        lowest = min(1 - miss[lab == g].mean() for lab in states.values() for g in np.unique(lab) if (lab == g).sum() >= 1000)
        rows.append({"method": f"{kind} {name}", "coverage": 1 - miss.mean(), "lowest state coverage": lowest, "mean margin dB": m.mean(),
                     "pinball loss x 1000": pinball(y[te], m, TAU).mean() * 1000})
headline = pd.DataFrame(rows).set_index("method")
headline.to_csv("CV_Results/server_side_margins.csv")
display(headline.round(4))

,coverage,lowest state coverage,mean margin dB,pinball loss x 1000
method,,,,
reg pooled,0.9985,0.9910,8.0105,88.3071
reg SF x hours,0.9930,0.9752,4.9580,64.7396
reg state table,0.9908,0.9847,4.4897,61.1740
reg recency,0.9974,0.9861,6.8078,78.6338
reg ACI,0.9905,0.9742,4.6596,65.7203
q pooled,0.9932,0.9866,4.5292,56.7936
q SF x hours,0.9924,0.9863,4.5215,56.8363
q state table,0.9916,0.9885,4.4381,56.4609
q recency,0.9915,0.9847,4.3496,56.3689


In [5]:
# Matched regime: one margin calibrated on the SF7-10 validation packets only, as the hold-out holds practically no SF11 or SF12 packets;
# hold-out coverage overall and in the states where a single margin is weakest
cal10 = np.flatnonzero(val & (sf <= 10))
matched_states = {"working hours": work[te] == 1, "volatile third": spread_groups(s20[te], cut) == 2, "SF9": sf[te] == 9}
rows = []
for name, b in (("previous-hour mean", np.zeros(len(d))), ("LightGBM regression, rung H4 without the channel", base["reg"])):
    m = shift_pooled(y[cal10] - b[cal10], TAU)
    covered = y[te] <= b[te] + m
    rows.append({"model": name, "margin dB": m, "mean margin above the anchor dB": (b[te] + m).mean(), "coverage": covered.mean(),
                 **{f"coverage, {k}": covered[v].mean() for k, v in matched_states.items()}})
matched = pd.DataFrame(rows).set_index("model")
matched.to_csv("CV_Results/server_side_matched_regime.csv")
display(matched.round(4))

,margin dB,mean margin above the anchor dB,coverage,"coverage, working hours","coverage, volatile third","coverage, SF9"
model,,,,,,
previous-hour mean,6.6780,6.6780,0.9914,0.9809,0.9610,0.9758
"LightGBM regression, rung H4 without the channel",5.2038,5.2162,0.9932,0.9852,0.9721,0.9862


In [6]:
# Estimators of adaptive data rate schemes on the link's earlier packets at the spreading factor of the predicted packet, as in notebook 17:
# the four filters and the best of the last 20 packets; the exponential average and the Kalman filter at the settings selected there on the validation folds
def gaussian_filter(w):
    """Mean of the window's values within one standard deviation of their mean (the Gaussian filter of G-ADR); at least one value always qualifies."""
    w = w[np.isfinite(w)]
    return w[np.abs(w - w.mean()) <= w.std(ddof=1)].mean()

def kalman(pl, minutes, q, r=1.0):
    """Kalman filter for a drifting level: the level moves by a random step whose variance grows with the minutes since the last packet."""
    pred = np.full(len(pl), np.nan)
    x, p = pl[0], r
    for i in range(1, len(pl)):
        p += q * minutes[i]
        pred[i] = x
        gain = p / (p + r)
        x += gain * (pl[i] - x)
        p *= 1 - gain
    return pred

tuning = pd.read_csv("CV_Results/adr_estimator_tuning.csv")
selected = tuning[(tuning.version == "same SF") & tuning.selected].set_index("estimator")["value"]
frame = d[["device_id", "SF"]].assign(v=PL, t=d["t"])
g = frame.groupby(["device_id", "SF"], sort=False)["v"]
last20 = lambda how: g.transform(lambda s: getattr(s.shift(1).rolling(20, min_periods=10), how)()).to_numpy()
minutes = frame.groupby(["device_id", "SF"], sort=False)["t"].diff().dt.total_seconds().to_numpy() / 60
code = frame.groupby(["device_id", "SF"], sort=False).ngroup().to_numpy()
kal = np.full(len(d), np.nan)
for c in np.unique(code):
    i = np.flatnonzero(code == c)
    kal[i] = kalman(PL[i], minutes[i], selected["Kalman filter"])
reference = "LightGBM on rung H4 without the channel"
est = {"mean of the last 20": last20("mean"),
       "Gaussian filter of the last 20": g.transform(lambda s: s.shift(1).rolling(20, min_periods=10).apply(gaussian_filter, raw=True)).to_numpy(),
       "exponential average": g.transform(lambda s: s.shift(1).ewm(alpha=selected["exponential average"], adjust=False).mean()).to_numpy(),
       "Kalman filter": kal,
       "best of the last 20": last20("min"),                    # lowest path loss = highest SNR, the standard rule
       reference: a + base["reg"]}
assert all(np.isfinite(p[val | ev]).all() for p in est.values()), "every estimator is defined on every validation and hold-out packet"
print(f"selected in notebook 17: exponential average weight {selected['exponential average']:g}, Kalman drift {selected['Kalman filter']:g} per minute")

selected in notebook 17: exponential average weight 0.2, Kalman drift 0.01 per minute


In [7]:
# Margins on the hold-out: the shift comes from the validation folds; RMSE of the level, coverage, margin above the anchor and pinball loss at 99 %,
# with 95 % intervals from resampling days, and the difference in pinball loss to LightGBM without the channel
g1_cal, g1_te = (sf[cal] - 7) * 2 + work[cal], (sf[te] - 7) * 2 + work[te]
dd = pd.factorize(day[te])[0]
ci = lambda cols, f: boot_stat(dd, np.column_stack(cols), f)
margin = {}
for name, p in est.items():
    b = p - a                                                    # threshold above the anchor before the shift
    score = PL[cal] - p[cal]
    margin[(name, "SF x hours")] = b[te] + shift_groups(score, g1_cal, g1_te, TAU)
    m = np.empty(len(te))
    for l in np.unique(link[te]):
        i = np.flatnonzero(link[te] == l)
        m[i] = aci(b[te][i], y[te][i], score, TAU, (1 - TAU) / GAMMA_DIVISOR)
    margin[(name, "ACI")] = m
rows = []
for (name, how), mm in margin.items():
    miss = y[te] > mm
    loss = pinball(y[te], mm, TAU)
    _, margin_lo, margin_hi = ci([mm, np.ones(len(te))], lambda T: T[:, 0] / T[:, 1])
    diff, lo, hi = ci([loss - pinball(y[te], margin[(reference, how)], TAU), np.ones(len(te))], lambda T: 1000 * T[:, 0] / T[:, 1])
    rows.append(dict(estimator=name, calibration=how, rmse=rmse(PL[te] - est[name][te]), coverage=1 - miss.mean(), margin=mm.mean(), margin_lo=margin_lo,
                     margin_hi=margin_hi, pinball=loss.mean() * 1000, pinball_minus_lightgbm=diff, lo=lo, hi=hi))
result = pd.DataFrame(rows)
result.to_csv("CV_Results/server_side_adr_estimators.csv", index=False)

# the classical estimators as in notebook 17, which scored them on the same packets
same = pd.read_csv("CV_Results/adr_estimators.csv").query("version == 'same SF'").set_index(["estimator", "calibration"])
classical = result[result.estimator != reference].set_index(["estimator", "calibration"])
cols = ["rmse", "coverage", "margin", "pinball"]
assert np.allclose(classical[cols], same.loc[classical.index, cols]), "the classical estimators differ from notebook 17"
lead = result.set_index(["estimator", "calibration"])
ref = lead.xs(reference, level="estimator")
lead["margin above LightGBM"] = lead.margin - ref.margin.reindex(lead.index.get_level_values("calibration")).to_numpy()
lead["RMSE above LightGBM"] = lead.rmse - ref.rmse.iloc[0]
print("hold-out, the link's earlier packets at the same SF: RMSE of the level estimate and, per calibration, coverage, mean margin above the anchor "
      "with its 95 % interval, pinball loss x 1000 and its difference to LightGBM without the channel")
display(lead.round(3))

hold-out, the link's earlier packets at the same SF: RMSE of the level estimate and, per calibration, coverage, mean margin above the anchor with its 95 % interval, pinball loss x 1000 and its difference to LightGBM without the channel


rmse  coverage  margin  \
estimator                               calibration                            
mean of the last 20                     SF x hours   2.189     0.993   5.795   
                                        ACI          2.189     0.991   5.584   
Gaussian filter of the last 20          SF x hours   2.235     0.993   6.056   
                                        ACI          2.235     0.991   5.879   
exponential average                     SF x hours   2.111     0.993   5.361   
                                        ACI          2.111     0.991   5.175   
Kalman filter                           SF x hours   2.104     0.993   5.309   
                                        ACI          2.104     0.991   5.155   
best of the last 20                     SF x hours   4.270     0.996   9.523   
                                        ACI          4.270     0.991   7.776   
LightGBM on rung H4 without the channel SF x hours   1.964     0.993   4.958   
                                        ACI          1.964     0.991   4.660   

                                                     margin_lo  margin_hi  \
estimator                               calibration                         
mean of the last 20                     SF x hours       5.603      5.978   
                                        ACI              5.377      5.818   
Gaussian filter of the last 20          SF x hours       5.842      6.256   
                                        ACI              5.643      6.158   
exponential average                     SF x hours       5.229      5.485   
                                        ACI              5.016      5.365   
Kalman filter                           SF x hours       5.189      5.424   
                                        ACI              4.996      5.345   
best of the last 20                     SF x hours       9.334      9.704   
                                        ACI              7.502      8.092   
LightGBM on rung H4 without the channel SF x hours       4.868      5.042   
                                        ACI              4.525      4.822   

                                                     pinball  \
estimator                               calibration            
mean of the last 20                     SF x hours    73.866   
                                        ACI           76.989   
Gaussian filter of the last 20          SF x hours    77.844   
                                        ACI           81.167   
exponential average                     SF x hours    68.738   
                                        ACI           71.412   
Kalman filter                           SF x hours    68.530   
                                        ACI           71.153   
best of the last 20                     SF x hours   109.639   
                                        ACI          101.128   
LightGBM on rung H4 without the channel SF x hours    64.740   
                                        ACI           65.720   

                                                     pinball_minus_lightgbm  \
estimator                               calibration                           
mean of the last 20                     SF x hours                    9.127   
                                        ACI                          11.268   
Gaussian filter of the last 20          SF x hours                   13.105   
                                        ACI                          15.447   
exponential average                     SF x hours                    3.999   
                                        ACI                           5.692   
Kalman filter                           SF x hours                    3.791   
                                        ACI                           5.432   
best of the last 20                     SF x hours                   44.900   
                                        ACI                          35.408   
LightGBM 

Reading of the results, 99 % target.

* Without the four channel inputs, LightGBM on rung H4 reaches 1.96 dB on the hold-out, against 1.77 dB with them (notebook 15).
* The quantile model needs 4.26 to 4.53 dB above the anchor with the five calibrations of the shift, against 3.78 to 3.98 dB with the channel (notebook 16), and covers 99.0 to 99.3 % of the hold-out and 98.5 to 98.9 % in its lowest state. The regression needs 4.96 dB with one shift per spreading factor x working hours (4.64 with the channel) and 4.66 dB with the adaptive shift (4.32).
* Matched regime: one margin calibrated on the SF7-10 validation packets sits 5.22 dB above the anchor on average (5.00 with the channel) and covers 99.3 % of the hold-out, 98.5 % in working hours, 97.2 % in the volatile third and 98.6 % at SF9. The quantile model with one shift saves 0.69 dB on it, against 1.03 dB with the channel.
* LightGBM keeps its lead over the estimators of adaptive data rate schemes at the same spreading factor. The four filters are 0.14 to 0.27 dB higher in RMSE and need 0.35 to 1.10 dB more margin with the shift per spreading factor x working hours and 0.50 to 1.22 dB more with the adaptive shift, against 0.67 to 1.42 and 0.84 to 1.56 dB with the channel (notebook 17). The 95 % intervals of the pinball-loss differences lie above zero for all four; the closest is the Kalman filter, 2.7 to 4.8 per thousand.
* The best of the last 20 packets needs 4.56 dB more than LightGBM with the shift per spreading factor x working hours and 3.12 dB more with the adaptive shift (4.89 and 3.46 with the channel). The rule as deployed, 10 dB above the best of the last 20 packets, sits 6.61 dB above the anchor and covers 98.3 %, 1.66 to 1.95 dB above LightGBM, which covers 99.1 to 99.3 %.